In [1]:
import json
import numpy as np
import librosa
import torch
import torch.nn as nn
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]
SR, N_MELS, TIME_FRAMES = 16000, 128, 128

WIN_S, HOP_S = 10.0, 5.0          # 10s window, 5s stride
FRAME_S      = WIN_S / 32          # 0.3125s per frame

COLORS = {"Scream":"#EF4444","Shout":"#F97316","Crying":"#F59E0B",
          "Explosion":"#EAB308","Gunshot":"#10B981","Glass":"#14B8A6",
          "Siren":"#6366F1","Alarm":"#8B5CF6"}

print("Device:", DEVICE)

Device: cuda


In [2]:
class CRNN_v3(nn.Module):
    def __init__(self, n, sed=True):
        super().__init__()
        self.sed = sed
        def blk(i,o,pool=(2,2)):
            return nn.Sequential(
                nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(pool), nn.Dropout2d(0.1))
        self.cnn  = nn.Sequential(blk(1,32,(2,2)), blk(32,64,(2,2)), blk(64,128,(2,1)))
        self.lstm = nn.LSTM(128*16, 128, batch_first=True, bidirectional=True,
                            num_layers=2, dropout=0.2)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(256, n)
    def forward(self, x):
        x = self.cnn(x); b,c,f,t = x.size()
        x = x.permute(0,3,1,2).contiguous().view(b,t,c*f)
        x,_ = self.lstm(x); x = self.drop(x)
        return self.fc(x) if self.sed else self.fc(x.mean(1))

In [3]:
model = CRNN_v3(8, sed=True).to(DEVICE)
model.load_state_dict(torch.load("best_audioset_sed_v3.pth"))
model.eval()

BEST_THR = {'Scream':0.82,'Shout':0.81,'Crying':0.92,'Explosion':0.75,
            'Gunshot':0.61,'Glass':0.82,'Siren':0.80,'Alarm':0.62}
SAFETY_THR = {**BEST_THR, "Gunshot":0.85, "Glass":0.90, "Shout":0.90}

print("Model loaded. Ready.")

Model loaded. Ready.


/tmp/ipykernel_1908728/3879675852.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_audioset_sed_v3.pth"))


In [4]:
def infer_probs(y, dur):
    """Sliding-window SED over full audio → averaged frame probabilities."""
    starts = np.arange(0, max(dur - WIN_S, 0) + HOP_S, HOP_S)
    if len(starts) == 0: starts = np.array([0.0])
    n_frames = int(np.ceil(dur / FRAME_S))
    acc  = np.zeros((n_frames, 8), dtype=np.float32)
    hits = np.zeros((n_frames, 1), dtype=np.float32) + 1e-8

    for s in starts:
        seg = y[int(s*SR): int((s+WIN_S)*SR)]
        if len(seg) < int(0.5*SR): continue
        if len(seg) < int(WIN_S*SR):
            seg = np.pad(seg, (0, int(WIN_S*SR)-len(seg)))
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=N_MELS))
        mel = (mel - mel.mean())/(mel.std()+1e-6)
        mel = np.pad(mel,((0,0),(0,TIME_FRAMES-mel.shape[1]))) if mel.shape[1]<TIME_FRAMES else mel[:,:TIME_FRAMES]
        x   = torch.tensor(mel[np.newaxis,np.newaxis], dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            with torch.amp.autocast("cuda"):
                p = torch.sigmoid(model(x))[0].float().cpu().numpy()
        f0 = int(s / FRAME_S)
        for k in range(p.shape[0]):
            fi = f0 + k
            if fi < n_frames:
                acc[fi] += p[k]; hits[fi] += 1
    return acc / hits

In [5]:
def probs_to_events(probs, thr, min_dur, dur):
    events = []
    for j, cls in enumerate(CLASSES):
        act, inev, st, sc = probs[:,j] > thr[cls], False, 0.0, []
        for t, a in enumerate(act):
            if a and not inev: st, inev, sc = t*FRAME_S, True, [probs[t,j]]
            elif a and inev:   sc.append(probs[t,j])
            elif not a and inev:
                if t*FRAME_S - st >= min_dur:
                    events.append((cls, st, t*FRAME_S, float(max(sc))))
                inev = False
        if inev and dur - st >= min_dur:
            events.append((cls, st, dur, float(max(sc))))
    events.sort(key=lambda e: e[1])
    return events

def make_warning(events, dur, mode):
    if not events:
        return f"## ✅  No distress content detected\n\n*Analysed {dur:.1f}s · mode: {mode}*"
    cls_set = sorted(set(e[0] for e in events))
    txt  = f"## ⚠️  CONTENT WARNING\n\n**This audio contains:** {', '.join(cls_set)}\n\n"
    txt += f"*Duration: {dur:.1f}s · {len(events)} events · mode: {mode}*\n\n"
    txt += "| # | Type | Start | End | Length | Conf |\n|---|---|---|---|---|---|\n"
    for i,(c,s,e,sc) in enumerate(events,1):
        txt += f"| {i} | **{c}** | {s:.1f}s | {e:.1f}s | {e-s:.1f}s | {sc:.2f} |\n"
    return txt

In [6]:
def make_plot(y, probs, events, thr, dur):
    t_ax = np.arange(len(probs)) * FRAME_S
    fig, ax = plt.subplots(2,1, figsize=(12,6.5), height_ratios=[1,1.2])

    mel_full = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS))
    ax[0].imshow(mel_full, aspect="auto", origin="lower", cmap="magma", extent=[0,dur,0,N_MELS])
    for c,s,e,sc in events:
        ax[0].axvspan(s, e, color=COLORS[c], alpha=0.25)
    ax[0].set_ylabel("Mel bin"); ax[0].set_title("Spectrogram + detected regions"); ax[0].set_xlim(0,dur)

    for j,c in enumerate(CLASSES):
        active = c in [e[0] for e in events]
        ax[1].plot(t_ax, probs[:,j], color=COLORS[c], lw=2.2 if active else 0.9,
                   alpha=1.0 if active else 0.25, label=c)
        ax[1].axhline(thr[c], color=COLORS[c], ls=":", lw=0.6, alpha=0.3)
    ax[1].set_xlim(0,dur); ax[1].set_ylim(0,1)
    ax[1].set_xlabel("Time (s)"); ax[1].set_ylabel("Probability")
    ax[1].set_title("Frame-level detection")
    ax[1].legend(ncol=8, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5,-0.22))
    ax[1].grid(alpha=0.25)
    plt.tight_layout()
    return fig

In [7]:
def analyse(audio_path, mode, min_dur):
    if audio_path is None:
        return None, "Upload an audio file first.", None

    thr  = SAFETY_THR if mode.startswith("Safety") else BEST_THR
    y, _ = librosa.load(audio_path, sr=SR, mono=True)
    dur  = len(y) / SR

    probs  = infer_probs(y, dur)
    events = probs_to_events(probs, thr, min_dur, dur)
    warn   = make_warning(events, dur, mode)
    fig    = make_plot(y, probs, events, thr, dur)
    table  = [[c, f"{s:.2f}", f"{e:.2f}", f"{e-s:.2f}", f"{sc:.3f}"] for c,s,e,sc in events]
    return fig, warn, table

In [8]:
with gr.Blocks(title="Trigger-Sense", theme=gr.themes.Soft()) as demo:
    gr.Markdown("Trigger-Sense")
    with gr.Row():
        with gr.Column(scale=1):
            audio = gr.Audio(type="filepath", label="Audio input")
            mode  = gr.Radio(["Balanced (best F1)","Safety (fewer false alarms)"],
                             value="Balanced (best F1)", label="Detection mode")
            mind  = gr.Slider(0.0, 2.0, value=0.3, step=0.1,
                              label="Min event duration (s)")
            btn   = gr.Button("Analyse", variant="primary", size="lg")
            gr.Markdown(f"**Classes:** {' · '.join(CLASSES)}")
        with gr.Column(scale=2):
            warn = gr.Markdown()
    plot  = gr.Plot(label="Detection timeline")
    table = gr.Dataframe(headers=["Class","Start (s)","End (s)","Duration (s)","Confidence"],
                         label="Detected events")
    btn.click(analyse, [audio, mode, mind], [plot, warn, table])

demo.launch(server_name="0.0.0.0", server_port=7860, share=False)

/tmp/ipykernel_1908728/2245454574.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Trigger-Sense", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
import json, shutil, os
import pandas as pd

# Load index if not already in memory
if "IDX" not in dir():
    with open("audioset_index.json") as f:
        IDX = json.load(f)

CLASSES = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]

df = pd.read_csv("audioset_v2_test.csv")
df["ytid"] = df["ytid"].str.strip()

os.makedirs("demo_clips", exist_ok=True)

print("=== SINGLE-CLASS DEMO CLIPS ===")
for c in CLASSES:
    match = df[df[c] == 1]
    if len(match) == 0:
        print(f"  {c}: none found"); continue
    row = match.iloc[0]
    dst = f"demo_clips/demo_{c}.wav"
    shutil.copy(IDX[row["ytid"]], dst)
    true = [x for x in CLASSES if row[x] == 1]
    print(f"  {dst}   (true: {true})")

print("\n=== MULTI-LABEL DEMO CLIP ===")
multi = df[df[CLASSES].sum(axis=1) >= 2]
if len(multi):
    row = multi.iloc[0]
    shutil.copy(IDX[row["ytid"]], "demo_clips/demo_multi.wav")
    print(f"  demo_clips/demo_multi.wav   (true: {[c for c in CLASSES if row[c]==1]})")

print("\nUpload any of these to the UI. Start with demo_Siren.wav (strongest class).")
print("Files are in: demo_clips/")

=== SINGLE-CLASS DEMO CLIPS ===


PermissionError: [Errno 13] Permission denied: 'demo_clips/demo_Scream.wav'